# Fetch L3 Ground Truth from Jira for `full_golden.parquet`

This notebook fetches Jira L3 ground truth and validates whether each Epic can be evaluated from its Stage-scoped L3 candidate set.

1. Load `full_golden.parquet`.
2. Extract every unique Epic key from `epic_keys`.
3. Fetch each Epic's configured L3 capabilities from Jira field `customfield_18603`.
4. Save one ground-truth row per Epic/L3 pair to `results/epic_l3_ground_truth_full_golden.xlsx`.

It does **not** run LLM evaluation.

In [ ]:
from pathlib import Path
import ast
import os
import re

import httpx
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display


def find_full_golden_file() -> Path:
    """Find full_golden.parquet in this notebook directory or its parent."""
    filename = "full_golden.parquet"
    for directory in (Path.cwd(), Path.cwd().parent):
        candidate = directory / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename} from {Path.cwd()}")


PARQUET_PATH = find_full_golden_file()
OUTPUT_PATH = (
    PARQUET_PATH.parent
    / "results"
    / "epic_l3_ground_truth_full_golden.xlsx"
)

print(f"Parquet: {PARQUET_PATH}")
print(f"Output:  {OUTPUT_PATH}")

## Extract unique Epic keys

In [ ]:
def parse_epic_keys(value) -> list[str]:
    """Parse Epic keys from a parquet cell."""
    if value is None:
        return []

    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()

    if isinstance(value, (list, tuple, set)):
        return [
            str(item).strip()
            for item in value
            if str(item).strip()
        ]

    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    text = str(value).strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        parsed = None

    if isinstance(parsed, (list, tuple, set)):
        return [
            str(item).strip()
            for item in parsed
            if str(item).strip()
        ]

    return re.findall(r"\b[A-Z][A-Z0-9_]*-\d+\b", text)


df = pd.read_parquet(PARQUET_PATH)

if "epic_keys" not in df.columns:
    raise KeyError("full_golden.parquet does not contain an 'epic_keys' column.")

all_epic_keys = sorted(
    {
        epic_key
        for value in df["epic_keys"]
        for epic_key in parse_epic_keys(value)
    }
)

print(f"Parquet rows: {len(df)}")
print(f"Unique Epics: {len(all_epic_keys)}")
print(all_epic_keys[:20])

## Jira L3 ground-truth fetch

In [ ]:
load_dotenv()

JIRA_BASE_URL = os.environ["JIRA_BASE_URL"].rstrip("/")
JIRA_TOKEN = os.environ["JIRA_TOKEN"]

HEADERS = {
    "Authorization": f"Bearer {JIRA_TOKEN}",
    "Accept": "application/json",
}

L3_CAP_FIELD_ID = "customfield_18603"


def get_epic_l3_cap(epic_key: str) -> dict:
    """Fetch one Epic's configured L3 capabilities from Jira."""
    response = httpx.get(
        f"{JIRA_BASE_URL}/rest/api/2/issue/{epic_key}",
        headers=HEADERS,
        params={"fields": f"summary,{L3_CAP_FIELD_ID}"},
        verify=False,
        timeout=60,
    )
    response.raise_for_status()

    issue = response.json()
    fields = issue.get("fields", {})
    raw_l3 = fields.get(L3_CAP_FIELD_ID) or []

    if not isinstance(raw_l3, list):
        raw_l3 = [raw_l3]

    l3_caps = []
    for item in raw_l3:
        if isinstance(item, dict):
            l3_caps.append(
                {
                    "value": item.get("value"),
                    "id": item.get("id"),
                }
            )
        else:
            l3_caps.append(
                {
                    "value": str(item),
                    "id": None,
                }
            )

    return {
        "epic_key": issue.get("key", epic_key),
        "epic_summary": fields.get("summary"),
        "l3_capabilities": l3_caps,
    }


def split_l3_value(value: str | None) -> tuple[str | None, str | None]:
    """Split 'Capability Name {CAP...}' into name and capability ID."""
    if not value:
        return None, None

    match = re.search(r"\{\s*(CAP\d+)\s*\}\s*$", value)
    if not match:
        return value.strip(), None

    return value[: match.start()].strip(), match.group(1)

## Fetch all Epics and save the GT workbook

In [ ]:
def export_jira_l3_ground_truth(
    epic_keys: list[str],
    output_path: str | Path = OUTPUT_PATH,
) -> pd.DataFrame:
    """Fetch Jira L3 values and export one row per Epic/L3 pair."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    rows = []
    unique_epics = sorted(set(epic_keys))

    for index, epic_key in enumerate(unique_epics, start=1):
        print(f"[{index}/{len(unique_epics)}] Fetching {epic_key}")

        try:
            result = get_epic_l3_cap(epic_key)
            l3_caps = result["l3_capabilities"]

            if not l3_caps:
                rows.append(
                    {
                        "epic_key": result["epic_key"],
                        "epic_summary": result["epic_summary"],
                        "l3_capability_id": None,
                        "l3_capability_name": None,
                        "jira_option_id": None,
                        "status": "no_l3_configured",
                        "error": None,
                    }
                )
                continue

            for l3_cap in l3_caps:
                capability_name, capability_id = split_l3_value(
                    l3_cap.get("value")
                )
                rows.append(
                    {
                        "epic_key": result["epic_key"],
                        "epic_summary": result["epic_summary"],
                        "l3_capability_id": capability_id,
                        "l3_capability_name": capability_name,
                        "jira_option_id": l3_cap.get("id"),
                        "status": "ok",
                        "error": None,
                    }
                )

        except Exception as exc:
            rows.append(
                {
                    "epic_key": epic_key,
                    "epic_summary": None,
                    "l3_capability_id": None,
                    "l3_capability_name": None,
                    "jira_option_id": None,
                    "status": "error",
                    "error": str(exc),
                }
            )

    ground_truth = pd.DataFrame(rows)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        ground_truth.to_excel(
            writer,
            sheet_name="jira_l3_ground_truth",
            index=False,
        )
        worksheet = writer.sheets["jira_l3_ground_truth"]
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        for column_cells in worksheet.columns:
            width = min(
                max(
                    len(str(cell.value or ""))
                    for cell in column_cells
                ) + 2,
                80,
            )
            worksheet.column_dimensions[
                column_cells[0].column_letter
            ].width = width

    print(f"Saved {len(ground_truth)} rows to: {output_path}")
    return ground_truth


gt_l3 = export_jira_l3_ground_truth(all_epic_keys)

## Fetch summary

In [ ]:
status_summary = (
    gt_l3.groupby("status")
    .size()
    .rename("row_count")
    .reset_index()
)

print(f"Unique Epics requested: {len(all_epic_keys)}")
print(f"Unique Epics returned:  {gt_l3['epic_key'].nunique()}")
print(f"Total GT rows:          {len(gt_l3)}")
print(
    "Configured L3 rows:    "
    f"{int((gt_l3['status'] == 'ok').sum())}"
)
print(
    "No L3 configured:      "
    f"{gt_l3.loc[gt_l3['status'] == 'no_l3_configured', 'epic_key'].nunique()}"
)
print(
    "Errors:                "
    f"{gt_l3.loc[gt_l3['status'] == 'error', 'epic_key'].nunique()}"
)

display(status_summary)
display(gt_l3.head(50))

## Validate GT against Stage candidate sets

An Epic is **VALID** only when every Jira GT L3 capability is present in the union of candidate L3 capabilities mapped to that Epic's Value Stream Stage(s). `full_golden.parquet` supplies the aligned `epic_keys` and `epic_vss`; `VSSCaprv (1).csv` supplies Stage → L3 candidates.

In [ ]:
def find_vsscap_file() -> Path:
    """Find the Stage-to-L3 capability mapping export."""
    filename = "VSSCaprv (1).csv"
    for directory in (Path.cwd(), Path.cwd().parent):
        candidate = directory / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename} from {Path.cwd()}")


def parse_aligned_values(value) -> list:
    """Return a parquet list-like cell as a Python list without changing order."""
    if value is None:
        return []

    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()

    if isinstance(value, (list, tuple)):
        return list(value)

    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    text = str(value).strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return [text]

    if isinstance(parsed, (list, tuple)):
        return list(parsed)
    return [parsed]


def extract_vss_ids(value) -> list[str]:
    """Extract all VSS IDs from one Epic's parquet VSS value."""
    if value is None:
        return []

    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()

    if isinstance(value, (list, tuple, set)):
        ids = []
        for item in value:
            ids.extend(extract_vss_ids(item))
        return list(dict.fromkeys(ids))

    return list(
        dict.fromkeys(
            match.upper()
            for match in re.findall(r"\bVSS\d+\b", str(value), flags=re.IGNORECASE)
        )
    )


if "epic_vss" not in df.columns:
    raise KeyError("full_golden.parquet does not contain an 'epic_vss' column.")

# epic_keys and epic_vss are aligned by position inside each parquet row.
epic_stage_ids: dict[str, set[str]] = {}
stage_alignment_errors: dict[str, str] = {}

for row_index, row in df.iterrows():
    epic_keys = parse_epic_keys(row["epic_keys"])
    epic_vss_values = parse_aligned_values(row["epic_vss"])

    if len(epic_keys) != len(epic_vss_values):
        for epic_key in epic_keys:
            stage_alignment_errors[epic_key] = (
                f"row {row_index}: epic_keys={len(epic_keys)} "
                f"but epic_vss={len(epic_vss_values)}"
            )
        continue

    for epic_key, epic_vss_value in zip(epic_keys, epic_vss_values):
        epic_stage_ids.setdefault(epic_key, set()).update(
            extract_vss_ids(epic_vss_value)
        )


VSSCAP_PATH = find_vsscap_file()
vsscap = pd.read_csv(
    VSSCAP_PATH,
    dtype=str,
    encoding="cp1252",
    encoding_errors="replace",
)

required_columns = {"Value Stream Stage ID", "Capability ID"}
missing_columns = required_columns.difference(vsscap.columns)
if missing_columns:
    raise KeyError(
        f"{VSSCAP_PATH.name} is missing columns: {sorted(missing_columns)}"
    )

vsscap = vsscap.copy()
vsscap["Value Stream Stage ID"] = (
    vsscap["Value Stream Stage ID"].fillna("").astype(str).str.strip().str.upper()
)
vsscap["Capability ID"] = (
    vsscap["Capability ID"].fillna("").astype(str).str.strip()
)
vsscap = vsscap.loc[
    vsscap["Value Stream Stage ID"].ne("")
    & vsscap["Capability ID"].ne("")
]

candidate_ids_by_stage = {
    stage_id: set(group["Capability ID"])
    for stage_id, group in vsscap.groupby("Value Stream Stage ID", sort=False)
}

gt_ok = gt_l3.loc[gt_l3["status"] == "ok"].copy()
gt_ok["l3_capability_id"] = (
    gt_ok["l3_capability_id"].fillna("").astype(str).str.strip()
)
gt_ok = gt_ok.loc[gt_ok["l3_capability_id"].ne("")]

gt_ids_by_epic = {
    epic_key: set(group["l3_capability_id"])
    for epic_key, group in gt_ok.groupby("epic_key", sort=False)
}

coverage_rows = []
for epic_key in all_epic_keys:
    gt_ids = gt_ids_by_epic.get(epic_key, set())
    stage_ids = epic_stage_ids.get(epic_key, set())

    candidate_ids = set()
    for stage_id in stage_ids:
        candidate_ids.update(candidate_ids_by_stage.get(stage_id, set()))

    missing_gt_ids = gt_ids - candidate_ids

    if epic_key in stage_alignment_errors:
        invalid_reason = "stage_alignment_error"
    elif not gt_ids:
        invalid_reason = "missing_ground_truth"
    elif not stage_ids:
        invalid_reason = "no_stage"
    elif not candidate_ids:
        invalid_reason = "no_candidates"
    elif missing_gt_ids:
        invalid_reason = "gt_not_fully_retrievable"
    else:
        invalid_reason = ""

    coverage_rows.append(
        {
            "epic_key": epic_key,
            "stage_ids": sorted(stage_ids),
            "gt_l3_ids": sorted(gt_ids),
            "candidate_l3_ids": sorted(candidate_ids),
            "missing_gt_l3_ids": sorted(missing_gt_ids),
            "valid": not invalid_reason,
            "invalid_reason": invalid_reason,
            "alignment_error": stage_alignment_errors.get(epic_key),
        }
    )

validity = pd.DataFrame(coverage_rows)
valid_epics = int(validity["valid"].sum())
invalid_epics = len(validity) - valid_epics

print(f"Total Epics:   {len(validity)}")
print(f"Valid Epics:   {valid_epics}")
print(f"Invalid Epics: {invalid_epics}")

print("\nInvalid breakdown:")
if invalid_epics:
    display(
        validity.loc[~validity["valid"], "invalid_reason"]
        .value_counts()
        .rename_axis("invalid_reason")
        .reset_index(name="epic_count")
    )
else:
    print("None")

print("\nInvalid Epic details:")
display(validity.loc[~validity["valid"]].reset_index(drop=True))


## Persist valid evaluation population

Save one row per valid Theme/Epic pair into the same GT workbook so E12-E14 can use an identical prevalidated population without repeating Jira Stage lookup or GT/candidate coverage checks.

In [ ]:
if "key" not in df.columns:
    raise KeyError("full_golden.parquet does not contain a 'key' Theme column.")

validity_by_epic = {
    row["epic_key"]: row
    for row in validity.to_dict(orient="records")
}

population_rows = []
for _, theme_row in df.iterrows():
    theme_key = str(theme_row.get("key") or "").strip()
    if not theme_key:
        continue

    for epic_key in parse_epic_keys(theme_row.get("epic_keys")):
        coverage = validity_by_epic.get(epic_key)
        if not coverage or not bool(coverage["valid"]):
            continue

        population_rows.append(
            {
                "theme_key": theme_key,
                "epic_key": epic_key,
                "stage_ids": json.dumps(coverage["stage_ids"]),
                "gt_l3_ids": json.dumps(coverage["gt_l3_ids"]),
                "candidate_l3_ids": json.dumps(coverage["candidate_l3_ids"]),
                "missing_gt_l3_ids": json.dumps(coverage["missing_gt_l3_ids"]),
                "valid": True,
                "invalid_reason": "",
            }
        )

evaluation_population = (
    pd.DataFrame(population_rows)
    .drop_duplicates(subset=["theme_key", "epic_key"], keep="first")
    .sort_values(["theme_key", "epic_key"], kind="stable")
    .reset_index(drop=True)
)

if evaluation_population["epic_key"].duplicated().any():
    duplicates = sorted(
        evaluation_population.loc[
            evaluation_population["epic_key"].duplicated(keep=False),
            "epic_key",
        ].unique()
    )
    raise ValueError(
        "A valid Epic is linked to multiple Themes; resolve before sampling: "
        f"{duplicates}"
    )

with pd.ExcelWriter(
    OUTPUT_PATH,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace",
) as writer:
    evaluation_population.to_excel(
        writer,
        sheet_name="evaluation_population",
        index=False,
    )

print(f"Saved valid evaluation population to: {OUTPUT_PATH}")
print(f"Valid Theme/Epic rows: {len(evaluation_population)}")
print(f"Valid unique Epics:     {evaluation_population['epic_key'].nunique()}")
print(f"Themes represented:     {evaluation_population['theme_key'].nunique()}")
display(evaluation_population.head(50))
